# H&M Personalized Fashion Recommendations

Two-stage recommender (candidate retrieval -> LightGBM LambdaRank) built with plain pandas / scikit-learn / LightGBM.
No GPU required. See `README.md` for the approach and `src/hm_reco/` for the implementation.

In [ ]:
import sys
sys.path.insert(0, "src")

import time
import pandas as pd

from hm_reco import data as hm_data
from hm_reco import features as hm_feat
from hm_reco import model as hm_model
from hm_reco import evaluation as hm_eval
from hm_reco import submission as hm_sub

DATA_DIR = "data"
OUTPUT_DIR = "outputs"

## 1. Load data

Downcasts dtypes and adds an integer `week` column to `transactions` so later steps stay memory-light.

In [ ]:
t0 = time.time()
ds = hm_data.load_dataset(DATA_DIR)
print(f"transactions: {ds.transactions.shape}, customers: {ds.customers.shape}, articles: {ds.articles.shape}")
print(f"n_weeks: {ds.n_weeks} (loaded in {time.time() - t0:.1f}s)")

## 2. Time-based validation split

We hold out the **last week** of the training data as validation (mirrors the real test week) and use the second-to-last
week as the training target, to avoid tuning on future information.

In [ ]:
VALIDATION_WEEK = ds.n_weeks - 1
TRAIN_WEEK = ds.n_weeks - 2
HISTORY_WEEKS = 12  # how far back feature/candidate functions may look

print(f"train target week: {TRAIN_WEEK}, validation week: {VALIDATION_WEEK}")

## 3. Build candidates + features for train / validation

Each call restricts itself to history strictly before its target week (no leakage), generates candidates
(repurchase, item-pairs, popular, age-bucket-popular), and attaches customer/article/customer-article features.

In [ ]:
t0 = time.time()
train_frame = hm_feat.build_training_frame(
    ds.transactions, ds.customers, ds.articles, TRAIN_WEEK,
    history_weeks=HISTORY_WEEKS, labeled=True,
)
print(f"train_frame: {train_frame.shape}, positive rate: {train_frame['label'].mean():.4%} ({time.time() - t0:.1f}s)")

In [ ]:
t0 = time.time()
val_frame = hm_feat.build_training_frame(
    ds.transactions, ds.customers, ds.articles, VALIDATION_WEEK,
    history_weeks=HISTORY_WEEKS, labeled=True,
)
print(f"val_frame: {val_frame.shape}, positive rate: {val_frame['label'].mean():.4%} ({time.time() - t0:.1f}s)")

### Candidate recall check

What fraction of actual validation-week purchases are even present in our candidate pool? This is the ceiling on the
final MAP@12 -- the ranker can only rank what recall found.

In [ ]:
actual_positive = val_frame["label"].sum()
actual_total = ds.transactions[ds.transactions["week"] == VALIDATION_WEEK].drop_duplicates(["customer_id", "article_id"]).shape[0]
print(f"candidate recall: {actual_positive / actual_total:.2%} ({actual_positive:,}/{actual_total:,})")
print(f"candidate precision: {actual_positive / len(val_frame):.2%} ({actual_positive:,}/{len(val_frame):,})")

## 4. Train the ranker

`LGBMRanker` with the LambdaRank objective, validated on held-out week with early stopping on MAP@12.

In [ ]:
model = hm_model.train_ranker(
    train_frame,
    eval_frame=val_frame,
    n_estimators=200,
    num_leaves=20,
    early_stopping_rounds=20,
)

## 5. Validate: compute MAP@12 on the held-out week

In [ ]:
scored = hm_model.score_candidates(model, val_frame)
predictions = hm_eval.predictions_from_scores(scored)
ground_truth = hm_eval.ground_truth_from_transactions(ds.transactions, VALIDATION_WEEK)

map12 = hm_eval.mean_average_precision(predictions, ground_truth)
print(f"Validation MAP@12: {map12:.5f}")

## 6. Retrain on the most recent week and predict for submission

For the actual submission we retrain using the *last available* week as the label week (closest to the real test
week), predict for every customer, and pad any customer with fewer than 12 candidates using last week's
most popular articles (there is no MAP@12 penalty for extra guesses).

In [ ]:
SUBMIT_TARGET_WEEK = ds.n_weeks  # one week beyond the data: what we're predicting for

sub_train_frame = hm_feat.build_training_frame(
    ds.transactions, ds.customers, ds.articles, ds.n_weeks - 1,
    history_weeks=HISTORY_WEEKS, labeled=True,
)
final_model = hm_model.train_ranker(sub_train_frame, n_estimators=150, num_leaves=20, early_stopping_rounds=None)

In [ ]:
predict_frame = hm_feat.build_training_frame(
    ds.transactions, ds.customers, ds.articles, SUBMIT_TARGET_WEEK,
    history_weeks=HISTORY_WEEKS,
    for_customers=ds.customers["customer_id"],
    labeled=False,
)
final_scored = hm_model.score_candidates(final_model, predict_frame)
final_predictions = hm_eval.predictions_from_scores(final_scored)
print(f"predicted for {len(final_predictions):,} customers")

## 7. Write submission.csv

In [ ]:
fallback_articles = (
    ds.transactions[ds.transactions["week"] == ds.n_weeks - 1]["article_id"]
    .value_counts()
    .head(12)
    .index
    .tolist()
)

submission = hm_sub.build_submission(
    final_predictions,
    ds.customers["customer_id"],
    ds.customer_id_map,
    fallback_articles,
)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
submission.to_csv(os.path.join(OUTPUT_DIR, "submission.csv"), index=False)

print(submission.shape)
submission.head()